# Monthly prefix analysis

Runs `analyze` on every `prefixes-*.parquet` file in each snapshot directory under `data/` and, for each snapshot, plots:

1. A bar chart of items with a processing baseline below 0510 per month.
2. A stacked bar chart of prefix counts per baseline per month.

In [1]:
import re
from pathlib import Path

import matplotlib.pyplot as plt

from mspc_sentinel_2_check import Analysis, analyze

In [ ]:
data_root = Path.cwd().parent / "data"
pattern = re.compile(r"^prefixes-(\d{4})-(\d{2})\.parquet$")
snapshots: dict[str, dict[str, Analysis]] = {}
for snapshot_dir in sorted(p for p in data_root.iterdir() if p.is_dir()):
    results: dict[str, Analysis] = {}
    for path in sorted(snapshot_dir.glob("prefixes-*.parquet")):
        match = pattern.match(path.name)
        if match is None:
            continue
        year, month = match.groups()
        results[f"{year}-{month}"] = analyze(str(path))
    snapshots[snapshot_dir.name] = results
{k: list(v) for k, v in snapshots.items()}

## Items below baseline 0510 per month

In [ ]:
for snapshot, results in snapshots.items():
    labels = list(results.keys())
    values = [r.below_baseline_0510 for r in results.values()]
    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.5), 4))
    ax.bar(labels, values)
    ax.set_xlabel("month")
    ax.set_ylabel("items below baseline 0510")
    ax.set_title(f"Sentinel-2 L2A items below baseline 0510 per month ({snapshot})")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    plt.show()

## Prefix count per baseline per month

In [ ]:
for snapshot, results in snapshots.items():
    labels = list(results.keys())
    baselines = sorted({b for r in results.values() for b in r.by_baseline})
    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.5), 4))
    bottom = [0] * len(labels)
    for baseline in baselines:
        counts = [results[m].by_baseline.get(baseline, 0) for m in labels]
        ax.bar(labels, counts, bottom=bottom, label=baseline)
        bottom = [b + c for b, c in zip(bottom, counts)]
    ax.set_xlabel("month")
    ax.set_ylabel("prefix count")
    ax.set_title(f"Sentinel-2 L2A prefix count per baseline per month ({snapshot})")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(title="baseline")
    fig.tight_layout()
    plt.show()